# Predoc Data Task Practice #1
#### Gavin Qu, 02-24-2026



This sample data task will use resource-level scheduled output data from the Electricity Reliability Council of Texas (ERCOT). For the first few questions, use the file `ercot_resource_output.csv`.

---

## Question 1

How many unique values does the variable `Resource Name` take in the data? The variable `QSE`? <br>
`Answer: There are 1121 unique values for Resource Name and 194 unique values for QSE.`

In [61]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/ercot_resource_output.csv")
print(f"data types: {df.info()}")

unique_val_resource = df['resource_name'].nunique()
print(f"Unique value count of [resource_name]: {unique_val_resource}")
unique_val_qse = df['qse'].nunique()
print(f"Unique value count of [QSE]: {unique_val_qse}")

<class 'pandas.DataFrame'>
RangeIndex: 3008438 entries, 0 to 3008437
Data columns (total 5 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   sced_time_stamp         str    
 1   qse                     str    
 2   resource_name           str    
 3   telemetered_net_output  float64
 4   resource_status         str    
dtypes: float64(1), str(4)
memory usage: 114.8 MB
data types: None
Unique value count of [resource_name]: 1121
Unique value count of [QSE]: 194


## Question 2

What is a QSE? Do a quick online search for this ERCOT acronym. Provide a brief (1-3 sentences) definition for QSE as used in ERCOT's market for electricity. <br>
`Answer: Qualified Scheduling Entity (QSE) are responsible for submitting bids/offers on behalf of resource entities or load serving entities such as retail electric providers and physical power plants. Essentially, QSEs acts ast the middlemen to bid or sell energy in the day-ahead market and the real-time market, balancing the supply and demand in the energy market. They are also responsible for financially settling with ERCOT(Electric Reliability Council of Texas).`

## Question 3

Find the set of unique QSE/Resource Name pairs. Answer the following questions.

### 3(a)

Is it ever the case that a single QSE is paired to multiple resource names? What might this indicate about the relationship between QSEs and Resource Names? What are the 10 largest QSEs in terms of the number of unique Resource Names they are paired to in the data?
<br>
`Answer: Yes, there are less unique QSEs comparing to resource names, this happens quite often by a quick visual of the dataset as well. It's indicative of a one to many relationship where QSEs manage multiple power plants.`

In [31]:
df_subset = df.loc[:, ['qse', 'resource_name']].drop_duplicates()
qse_counts = df_subset.value_counts(['qse'])
print(qse_counts.head(10))

qse   
QTENSK    173
QLUMN      75
QNRGTX     50
QCALP      44
QECNR      40
QAEN       34
QLCRA      32
QCPSE      25
QTEN23     24
QSHEL2     22
Name: count, dtype: int64


### 3(b)

Is it ever the case that a single Resource Name is paired to more than one QSE in the data? For how many Resource Names is this true for? Why might a single Resource Name pair with multiple QSEs in the data? **Hint:** Look at how pairs change over time. <br>
`Answer: Yes, there are 6 of those instances. This could happen because of buyer and seller are different for the same resources or ownership change of the power plants, or that qse contracts expired and new broker was found which could explain the same switching date for the same QSEs.`

In [47]:
resource_counts = df_subset.value_counts(['resource_name'])
multi_resource = resource_counts[resource_counts > 1]
multi_resource

resource_name  
MUSTNGCK_BES1      2
MUSTNGCK_SOLAR1    2
MUSTNGCK_SOLAR2    2
STAM_SLR_BESS1     2
STAM_SLR_SOLAR1    2
STAM_SLR_SOLAR2    2
Name: count, dtype: int64

In [51]:
# The pandas equivalent of R's: df %>% group_by() %>% filter() %>% arrange()
multi_qse_pairs = (
    df_subset.groupby('resource_name')
    .filter(lambda x: x['qse'].nunique() > 1)
    .sort_values('resource_name')
)

print(multi_qse_pairs)

            qse    resource_name   sced_time_stamp
566      QENEL5    MUSTNGCK_BES1  01/22/2023 00:00
279638   QENE10    MUSTNGCK_BES1  01/25/2023 00:00
567      QENEL5  MUSTNGCK_SOLAR1  01/22/2023 00:00
279639   QENE10  MUSTNGCK_SOLAR1  01/25/2023 00:00
568      QENEL5  MUSTNGCK_SOLAR2  01/22/2023 00:00
279640   QENE10  MUSTNGCK_SOLAR2  01/25/2023 00:00
792      QENEL5   STAM_SLR_BESS1  01/22/2023 00:00
2253533  QENE12   STAM_SLR_BESS1  02/15/2023 00:00
793      QENEL5  STAM_SLR_SOLAR1  01/22/2023 00:00
2253534  QENE12  STAM_SLR_SOLAR1  02/15/2023 00:00
794      QENEL5  STAM_SLR_SOLAR2  01/22/2023 00:00
2253535  QENE12  STAM_SLR_SOLAR2  02/15/2023 00:00


## Question 4

Now turn to `resource_type.csv`.

### 4(a)

How many unique, non-missing values does `Resource Type` take? Can you find definitions for them? (No need to define all of them, just attempt a few.)
`Answer: There are 15 unique, non-missing values for resource type, wind, solar, hydroelectric, nuclear, just to name a few. `

In [60]:
df_type = pd.read_csv("./data/ercot_resource_types.csv")
print(f"{df_type.info()}\n")
print(f"unique count of resource type: {df_type['resource_type'].nunique()}\n")
print(f"sample of unique resource type: {df_type['resource_type'].drop_duplicates().head(15)}")

<class 'pandas.DataFrame'>
RangeIndex: 1121 entries, 0 to 1120
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   resource_name  1121 non-null   str  
 1   resource_type  1117 non-null   str  
dtypes: str(2)
memory usage: 17.6 KB
None

unique count of resource type: 15

sample of unique resource type: 0         DSL
3      SCGT90
6        WIND
11     PWRSTR
15      HYDRO
17     CCGT90
27       PVGR
47     SCLE90
111     GSREH
148    CCLE90
153     CLLIG
179     GSSUP
206       NUC
257    GSNONR
293     RENEW
Name: resource_type, dtype: str


### 4(b)

Are there any empty strings in the resource type column? Which resource names are missing their type? Can you guess what the missing values should be? Fill in the missing values with your guesses (you will carry your filled in guesses for the remainder of the data task).
`Answer: There are four resource names where resource type is missing, they are solar and wind companies.`

In [62]:
empty_mask = (df_type['resource_type'].isna()) | (df_type['resource_type'] == '')
print(df_type[empty_mask])

       resource_name resource_type
341  GALLOWAY_SOLAR1           NaN
711  ROSELAND_SOLAR3           NaN
791  SSPURTWO_WIND_1           NaN
814   SWEETWN2_WND24           NaN


In [64]:
# Impute missing string based on suffix in resource name
df_type.loc[df_type['resource_name'].str.contains('SOLAR', na=False), 'resource_type'] = 'PVGR'
df_type.loc[df_type['resource_name'].str.contains('WIND|WND', na=False), 'resource_type'] = 'WIND'

# Verify the nan are gone
print(df_type[df_type['resource_type'].isna()])

Empty DataFrame
Columns: [resource_name, resource_type]
Index: []
